## Reproducibility & Libraries

In [9]:
import os
import random
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.preprocessing import StandardScaler

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(seed=SEED)

## Data Load

In practical implementation,setup might produce upto **500 readings/second**. <br>Hence, for faster processing and efficient memory usage we used **C engine** to enable **low memory option** & converted all values to **int32**.

In [10]:
df_path = "data/data.csv"

df = pd.read_csv(
    filepath_or_buffer=df_path,
    engine='c',
    low_memory=True,
    dtype={'value': 'Int32'},  
    parse_dates=['timestamp'],
    on_bad_lines='skip'
)

## Data Preprocessing

Quicksort has **low time complexity and space complexity** which makes it suitable for EEG application.<br>
TIME COMPLEXITY: **O(nlogn)** <br> 
SPACE COMPLEXITY: **O(logn)**

In [ ]:
df = df.sort_values(
    by='timestamp', 
    kind='quicksort'
).reset_index(drop=True)

As per research, **2s Time Window** length has the best performance on emotion recognition and is the most suitable to be used in EEG feature extraction for emotion recognition. 

In [ ]:
df['timestamp_window'] = df['timestamp'].dt.floor(
    freq='2s', 
    ambiguous='raise', 
    nonexistent='raise'
)

Combining all the records obtained in each 2s TW interval to obtain their **fluctuation pattern**.

In [11]:
grouped_windows = df.groupby(
    by='timestamp_window',
    sort=True,
    dropna=True
)

df['reading_index'] = grouped_windows.cumcount(ascending=True) + 1

df_pivot = df.pivot(
    columns='reading_index', 
    index='timestamp_window', 
    values='value'
)

df_pivot = df_pivot.astype(dtype=float, errors='raise')

df_pivot = df_pivot.ffill(axis=1)

df_pivot = df_pivot.add_prefix(
    prefix='signal_value_',
    axis='columns'
)

df_pivot = df_pivot.reset_index()

## Feature Engineering

**Alpha waves(8 to 13 Hz)** show the resting phase in awakened state.<br>
**Beta waves(14 to 30 Hz)** represent the state when senses are stimulated and mental activity is high.<br>
**Gamma waves(over 30 Hz)** are seen when the brain alertness is extremely high.<br> 
As per observation, **more fluctuation cycles are noticed in attentive state** like Beta and Gamma than in relaxed state like Alpha.<br>
Hence,**ZCR(zero-crossing rate)** and **variance** can be used as important features.

In [12]:
signal_columns = df_pivot.drop(columns=['timestamp_window'])

X = signal_columns.values
X_center = X - np.nanmean(X, axis=1, keepdims=True)

features = pd.DataFrame({
    'total_2s_variance': np.nanstd(X, axis=1, ddof=1),
    'total_2s_zcr': np.sum(np.diff(np.signbit(X_center), axis=1), axis=1) / (X.shape[1] - 1)
}, index=df_pivot.index)

for s, name in [(2, '1s'), (4, '0_5s')]:
    features[f'avg_{name}_variance'] = np.mean([np.nanstd(w, axis=1, ddof=1) for w in np.array_split(X, s, axis=1)], axis=0)
    features[f'avg_{name}_zcr'] = np.mean([np.sum(np.diff(np.signbit(w), axis=1), axis=1) / (w.shape[1] - 1) for w in np.array_split(X_center, s, axis=1)], axis=0)

features = features.replace([np.inf, -np.inf], np.nan).dropna(axis=0, how='any')

df_pivot = df_pivot.loc[features.index].reset_index(drop=True)
features = features.reset_index(drop=True)

## Model training

For **Model selection**, we tested unsupervised learning methods like KMeans, Gaussian Mixture Model, Agglomerative clustering, DBSCAN.<br>
Out of which **DBSCAN** performed highest with **Silhouette Coefficient of 0.53**.<br>
Optuna tuned parameters : {"EPS": 4.494850, "MIN_SAMPLES": 6, "METRIC": "euclidean"}

In [13]:
EPS, MIN_SAMPLES, METRIC = 4.494850, 6, 'euclidean'

features_scaled = StandardScaler().fit_transform(features)

labels = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES, metric=METRIC).fit_predict(features_scaled)

## Model Evaluation

In [16]:
raw_labels = np.unique(labels)
valid_clusters = raw_labels[raw_labels != -1]

if len(raw_labels) < 2 or len(valid_clusters) < 1:
    print("Execution Status      : failed")
else:
    print(f"Silhouette Coefficient     : {silhouette_score(features_scaled, labels, metric=METRIC):.6f}")
    print(f"Davies-Bouldin Index       : {davies_bouldin_score(features_scaled, labels):.3f}")
    print(f"Calinski-Harabasz Score    : {calinski_harabasz_score(features_scaled, labels):.3f}")
features['cluster_id'] = labels

Silhouette Coefficient     : 0.916088
Davies-Bouldin Index       : 0.114
Calinski-Harabasz Score    : 1210.847


## Result mapping

**Beta, Gamma:** Attentive State<br>
**Alpha:** Relaxed State

In [18]:
if len(valid_clusters) >= 2:
    c_centers = {0: np.mean(features_scaled[labels == 0], axis=0), 1: np.mean(features_scaled[labels == 1], axis=0)}
    clean_labels = labels.copy()
    
    for idx in np.where(labels == -1)[0]:
        clean_labels[idx] = min(c_centers, key=lambda c: np.linalg.norm(features_scaled[idx] - c_centers[c]))
else:
    clean_labels = np.where(labels == -1, 0, labels)

df_pivot['state'] = clean_labels

features['cluster_id'] = clean_labels
zcr_means = features.groupby('cluster_id')['total_2s_zcr'].mean()

label_0_meaning = "beta, gamma" if zcr_means.get(0, 0) > zcr_means.get(1, 0) else "alpha"
label_1_meaning = "beta, gamma" if label_0_meaning == "alpha" else "alpha"

print(f"0: {label_0_meaning}")
print(f"1: {label_1_meaning}")

0: beta, gamma
1: alpha


**REFERENCES:**<br>
**[1]** Gnanapandithan K, Sharma R, Sharma S. Gabapentin toxicity: A review of the literature. *Cureus*. 2022 Jun 15;14(6):e26001. doi: 10.7759/cureus.26001. Available from: [https://pmc.ncbi.nlm.nih.gov/articles/PMC9269830/](https://pmc.ncbi.nlm.nih.gov/articles/PMC9269830/)<br>
**[2]** Thapa S, Singh J. Gabapentin. [Updated 2024 Jan 17]. In: StatPearls [Internet]. Treasure Island (FL): StatPearls Publishing; 2024 Jan-. Available from: [https://www.ncbi.nlm.nih.gov/books/NBK563104/](https://www.google.com/search?q=https%3A%2F%2Fwww.ncbi.nlm.nih.gov%2Fbooks%2FNBK563104%2F)